In [1]:
# -*- coding: utf-8 -*-
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import pandas as pd
import pickle
import copy
import json
import time
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')

class Config:
    IMMATURE_DIR = r"E:\TSG\jupyterlab\machine learning image\augmented_dataset\immature"
    MATURE_DIR = r"E:\TSG\jupyterlab\machine learning image\augmented_dataset\mature"
    
    SEED = 42
    BATCH_SIZE = 32
    NUM_EPOCHS = 100
    LEARNING_RATE = 0.001
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    STEP_SIZE = 30
    GAMMA = 0.1
    
    SAVE_DIR = "./saved_models"
    MODEL_NAME = "parallel_resnet18"
    
    NUM_CLASSES = 2
    TEST_SIZE = 0.2
    N_FOLDS = 5
    IMAGE_SIZE = 224
    
    @classmethod
    def save_config(cls, save_path):
        config_dict = {
            'IMMATURE_DIR': cls.IMMATURE_DIR,
            'MATURE_DIR': cls.MATURE_DIR,
            'SEED': cls.SEED,
            'BATCH_SIZE': cls.BATCH_SIZE,
            'NUM_EPOCHS': cls.NUM_EPOCHS,
            'LEARNING_RATE': cls.LEARNING_RATE,
            'MOMENTUM': cls.MOMENTUM,
            'WEIGHT_DECAY': cls.WEIGHT_DECAY,
            'STEP_SIZE': cls.STEP_SIZE,
            'GAMMA': cls.GAMMA,
            'SAVE_DIR': cls.SAVE_DIR,
            'MODEL_NAME': cls.MODEL_NAME,
            'NUM_CLASSES': cls.NUM_CLASSES,
            'TEST_SIZE': cls.TEST_SIZE,
            'N_FOLDS': cls.N_FOLDS,
            'IMAGE_SIZE': cls.IMAGE_SIZE
        }
        with open(save_path, 'w') as f:
            json.dump(config_dict, f, indent=4)
    
    @classmethod
    def load_config(cls, config_path):
        with open(config_path, 'r') as f:
            config_dict = json.load(f)
        for key, value in config_dict.items():
            setattr(cls, key, value)

def check_gpu_available():
    if not torch.cuda.is_available():
        print("Warning: No GPU detected, training with CPU (very slow)")
        print("Recommendation: Install CUDA, cuDNN and GPU version of PyTorch")
        return False
    
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    return True

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

class ParallelResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_ftrs = base.fc.in_features
        base.fc = nn.Identity()
        self.branch1 = copy.deepcopy(base)
        self.branch2 = copy.deepcopy(base)
        self.fc = nn.Linear(num_ftrs * 2, num_classes)

    def forward(self, x):
        f1 = self.branch1(x)
        f2 = self.branch2(x)
        feats = torch.cat([f1, f2], dim=1)
        out = self.fc(feats)
        return out

class DataManager:
    def __init__(self, immature_dir, mature_dir):
        self.immature_dir = immature_dir
        self.mature_dir = mature_dir
        self.train_paths = None
        self.train_labels = None
        self.test_paths = None
        self.test_labels = None
        
    def load_all_data(self):
        immature_paths = []
        mature_paths = []
        
        for img_name in os.listdir(self.immature_dir):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                immature_paths.append(os.path.join(self.immature_dir, img_name))
        
        for img_name in os.listdir(self.mature_dir):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                mature_paths.append(os.path.join(self.mature_dir, img_name))
        
        all_paths = immature_paths + mature_paths
        all_labels = [0] * len(immature_paths) + [1] * len(mature_paths)
        
        print(f"Immature images: {len(immature_paths)}")
        print(f"Mature images: {len(mature_paths)}")
        print(f"Total images: {len(all_paths)}")
        
        return all_paths, all_labels
    
    def split_data(self, test_size=0.2, random_state=42):
        all_paths, all_labels = self.load_all_data()
        
        self.train_paths, self.test_paths, self.train_labels, self.test_labels = train_test_split(
            all_paths, all_labels, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=all_labels
        )
        
        print(f"Train set size: {len(self.train_paths)}")
        print(f"Test set size: {len(self.test_paths)}")
        
        return self.train_paths, self.test_paths, self.train_labels, self.test_labels
    
    def get_transforms(self):
        train_transform = transforms.Compose([
            transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        
        val_transform = transforms.Compose([
            transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        
        return train_transform, val_transform
    
    def create_datasets(self):
        train_transform, val_transform = self.get_transforms()
        
        train_dataset = CustomDataset(self.train_paths, self.train_labels, train_transform)
        test_dataset = CustomDataset(self.test_paths, self.test_labels, val_transform)
        
        return train_dataset, test_dataset

class ModelManager:
    def __init__(self, save_dir=Config.SAVE_DIR, model_name=Config.MODEL_NAME):
        self.save_dir = Path(save_dir)
        self.model_name = model_name
        self.save_dir.mkdir(parents=True, exist_ok=True)
        
        self.model_path = self.save_dir / f"{model_name}_model.pth"
        self.best_model_path = self.save_dir / f"{model_name}_best.pth"
        self.checkpoint_path = self.save_dir / f"{model_name}_checkpoint.pth"
        self.info_path = self.save_dir / f"{model_name}_info.pkl"
        self.config_path = self.save_dir / f"{model_name}_config.json"
        
    def create_model(self, num_classes=Config.NUM_CLASSES):
        model = ParallelResNet18(num_classes=num_classes)
        return model
    
    def save_model(self, model, epoch, optimizer, scheduler, 
                   fold_results=None, test_results=None, train_results=None):
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'fold_results': fold_results,
            'test_results': test_results,
            'train_results': train_results,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }, self.checkpoint_path)
        
        print(f"✓ Model checkpoint saved to: {self.checkpoint_path}")
    
    def save_best_model(self, model):
        torch.save(model.state_dict(), self.best_model_path)
        print(f"✓ Best model saved to: {self.best_model_path}")
    
    def save_model_info(self, data_manager, fold_results, test_results, train_results):
        info = {
            'train_paths': data_manager.train_paths if data_manager else None,
            'test_paths': data_manager.test_paths if data_manager else None,
            'train_labels': data_manager.train_labels if data_manager else None,
            'test_labels': data_manager.test_labels if data_manager else None,
            'class_names': ['immature', 'mature'],
            'normalization_mean': [0.485, 0.456, 0.406],
            'normalization_std': [0.229, 0.224, 0.225],
            'fold_results': fold_results,
            'test_results': test_results,
            'train_results': train_results
        }
        
        with open(self.info_path, 'wb') as f:
            pickle.dump(info, f)
        
        print(f"✓ Model info saved to: {self.info_path}")
    
    def load_model(self, device, load_checkpoint=True):
        if load_checkpoint and self.checkpoint_path.exists():
            checkpoint = torch.load(self.checkpoint_path, map_location=device)
            model = self.create_model()
            model.load_state_dict(checkpoint['model_state_dict'])
            model = model.to(device)
            
            print(f"✓ Loaded model from checkpoint (epoch {checkpoint['epoch']})")
            print(f"  Saved time: {checkpoint.get('timestamp', 'Unknown')}")
            
            return model, checkpoint
        elif self.best_model_path.exists():
            model = self.create_model()
            model.load_state_dict(torch.load(self.best_model_path, map_location=device))
            model = model.to(device)
            print(f"✓ Loaded best model")
            return model, None
        else:
            print("⚠ No saved model found, creating new model")
            model = self.create_model()
            model = model.to(device)
            return model, None
    
    def load_model_info(self):
        if self.info_path.exists():
            with open(self.info_path, 'rb') as f:
                info = pickle.load(f)
            print(f"✓ Loaded model info")
            return info
        else:
            print("⚠ No model info file found")
            return None

class TrainingUtils:
    @staticmethod
    def calculate_metrics(all_labels, all_predictions):
        accuracy = np.mean(np.array(all_labels) == np.array(all_predictions))
        precision = precision_score(all_labels, all_predictions, average='weighted')
        recall = recall_score(all_labels, all_predictions, average='weighted')
        f1 = f1_score(all_labels, all_predictions, average='weighted')
        precision_per_class = precision_score(all_labels, all_predictions, average=None)
        recall_per_class = recall_score(all_labels, all_predictions, average=None)
        f1_per_class = f1_score(all_labels, all_predictions, average=None)
        cm = confusion_matrix(all_labels, all_predictions)
        report = classification_report(all_labels, all_predictions, 
                                      target_names=['immature', 'mature'], 
                                      output_dict=True)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'precision_per_class': precision_per_class,
            'recall_per_class': recall_per_class,
            'f1_per_class': f1_per_class,
            'confusion_matrix': cm,
            'classification_report': report
        }
    
    @staticmethod
    def train_epoch(model, dataloader, criterion, optimizer, device):
        model.train()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        progress_bar = tqdm(dataloader, desc='Training')
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            batch_acc = (predicted == labels).sum().item() / labels.size(0)
            progress_bar.set_postfix({'Loss': running_loss/len(dataloader), 'Acc': batch_acc})
        
        epoch_loss = running_loss / len(dataloader)
        metrics = TrainingUtils.calculate_metrics(all_labels, all_predictions)
        metrics['loss'] = epoch_loss
        
        return epoch_loss, metrics
    
    @staticmethod
    def evaluate_model(model, dataloader, criterion, device, dataset_name="Dataset"):
        model.eval()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in dataloader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_loss = running_loss / len(dataloader)
        metrics = TrainingUtils.calculate_metrics(all_labels, all_predictions)
        metrics['loss'] = epoch_loss
        
        return epoch_loss, metrics
    
    @staticmethod
    def print_detailed_metrics(metrics, dataset_name="Dataset"):
        print(f"\n{dataset_name} Detailed Metrics:")
        print("-" * 50)
        print(f"Loss: {metrics['loss']:.4f}")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1-Score: {metrics['f1']:.4f}")
        
        print(f"\nPer-class Metrics:")
        print(f"  Immature (0): Precision={metrics['precision_per_class'][0]:.4f}, "
              f"Recall={metrics['recall_per_class'][0]:.4f}, F1={metrics['f1_per_class'][0]:.4f}")
        print(f"  Mature (1): Precision={metrics['precision_per_class'][1]:.4f}, "
              f"Recall={metrics['recall_per_class'][1]:.4f}, F1={metrics['f1_per_class'][1]:.4f}")
        
        print(f"\nConfusion Matrix:")
        print(metrics['confusion_matrix'])

class Trainer:
    def __init__(self, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.data_manager = None
        self.model_manager = None
        self.utils = TrainingUtils()
        
    def setup(self, retrain=False):
        print("=" * 80)
        print("Parallel ResNet18 Transfer Learning Training System")
        print("=" * 80)
        
        check_gpu_available()
        
        set_seed(Config.SEED)
        
        print(f"\nUsing device: {self.device}")
        print(f"Save directory: {Config.SAVE_DIR}")
        
        self.data_manager = DataManager(Config.IMMATURE_DIR, Config.MATURE_DIR)
        self.model_manager = ModelManager()
        
        Config.save_config(self.model_manager.config_path)
        
        if not retrain and self.model_manager.checkpoint_path.exists():
            print("\n⚠ Saved model checkpoint found")
            response = input("Resume training from checkpoint? (y/n): ").lower()
            if response == 'y':
                return 'resume'
        
        return 'new'
    
    def kfold_cross_validation(self, train_dataset):
        print("\n" + "="*60)
        print("Starting 5-fold cross validation...")
        print("="*60)
        
        kfold = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
        fold_results = []
        
        for fold, (train_idx, val_idx) in enumerate(kfold.split(train_dataset)):
            print(f"\n{'='*60}")
            print(f"Fold {fold+1}/{Config.N_FOLDS}")
            print(f"{'='*60}")
            
            train_subsampler = SubsetRandomSampler(train_idx)
            val_subsampler = SubsetRandomSampler(val_idx)
            
            train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, 
                                      sampler=train_subsampler)
            val_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, 
                                    sampler=val_subsampler)
            
            model = self.model_manager.create_model()
            model = model.to(self.device)
            
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.SGD(model.parameters(), lr=Config.LEARNING_RATE, 
                                  momentum=Config.MOMENTUM, weight_decay=Config.WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=Config.STEP_SIZE, 
                                                  gamma=Config.GAMMA)
            
            best_val_acc = 0
            best_model_state = None
            best_train_metrics = None
            best_val_metrics = None
            best_train_loss = None
            
            for epoch in range(Config.NUM_EPOCHS):
                print(f"\nEpoch {epoch+1}/{Config.NUM_EPOCHS}")
                
                train_loss, train_metrics = self.utils.train_epoch(
                    model, train_loader, criterion, optimizer, self.device)
                
                val_loss, val_metrics = self.utils.evaluate_model(
                    model, val_loader, criterion, self.device, "Validation")
                
                scheduler.step()
                
                print(f"Train - Loss: {train_loss:.4f}, Acc: {train_metrics['accuracy']:.4f}, "
                      f"F1: {train_metrics['f1']:.4f}")
                print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_metrics['accuracy']:.4f}, "
                      f"F1: {val_metrics['f1']:.4f}")
                
                if val_metrics['accuracy'] > best_val_acc:
                    best_val_acc = val_metrics['accuracy']
                    best_model_state = model.state_dict().copy()
                    best_train_metrics = train_metrics
                    best_val_metrics = val_metrics
                    best_train_loss = train_loss
            
            fold_results.append({
                'fold': fold + 1,
                'best_val_acc': best_val_acc,
                'val_loss': best_val_metrics['loss'],
                'val_accuracy': best_val_metrics['accuracy'],
                'val_precision': best_val_metrics['precision'],
                'val_recall': best_val_metrics['recall'],
                'val_f1': best_val_metrics['f1'],
                'val_precision_per_class': best_val_metrics['precision_per_class'],
                'val_recall_per_class': best_val_metrics['recall_per_class'],
                'val_f1_per_class': best_val_metrics['f1_per_class'],
                'train_loss': best_train_loss,
                'train_accuracy': best_train_metrics['accuracy'],
                'train_precision': best_train_metrics['precision'],
                'train_recall': best_train_metrics['recall'],
                'train_f1': best_train_metrics['f1'],
                'model_state': best_model_state
            })
        
        print("\n" + "="*60)
        print("Cross Validation Results Summary:")
        print("="*60)
        for result in fold_results:
            print(f"Fold {result['fold']}: "
                  f"Val Acc = {result['best_val_acc']:.4f}, "
                  f"Val F1 = {result['val_f1']:.4f}, "
                  f"Val Loss = {result['val_loss']:.4f}")
        
        avg_val_acc = np.mean([r['best_val_acc'] for r in fold_results])
        avg_val_f1 = np.mean([r['val_f1'] for r in fold_results])
        avg_val_loss = np.mean([r['val_loss'] for r in fold_results])
        
        print(f"\nAverage Validation Accuracy: {avg_val_acc:.4f}")
        print(f"Average Validation F1 Score: {avg_val_f1:.4f}")
        print(f"Average Validation Loss: {avg_val_loss:.4f}")
        
        return fold_results
    
    def train_final_model(self, train_dataset, test_dataset, start_epoch=0):
        print("\n" + "="*60)
        print("Training final model on entire training set...")
        print("="*60)
        
        train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        if start_epoch > 0:
            model, checkpoint = self.model_manager.load_model(self.device)
            if checkpoint:
                optimizer = optim.SGD(model.parameters(), lr=Config.LEARNING_RATE,
                                    momentum=Config.MOMENTUM, weight_decay=Config.WEIGHT_DECAY)
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                
                if checkpoint['scheduler_state_dict']:
                    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=Config.STEP_SIZE,
                                                        gamma=Config.GAMMA)
                    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                else:
                    scheduler = None
        else:
            model = self.model_manager.create_model()
            model = model.to(self.device)
            optimizer = optim.SGD(model.parameters(), lr=Config.LEARNING_RATE,
                                momentum=Config.MOMENTUM, weight_decay=Config.WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=Config.STEP_SIZE,
                                                gamma=Config.GAMMA)
        
        criterion = nn.CrossEntropyLoss()
        
        best_test_acc = 0
        best_test_metrics = None
        best_train_metrics = None
        best_train_loss = None
        
        for epoch in range(start_epoch, Config.NUM_EPOCHS):
            print(f"\nFinal Model - Epoch {epoch+1}/{Config.NUM_EPOCHS}")
            
            train_loss, train_metrics = self.utils.train_epoch(
                model, train_loader, criterion, optimizer, self.device)
            
            test_loss, test_metrics = self.utils.evaluate_model(
                model, test_loader, criterion, self.device, "Test")
            
            if scheduler:
                scheduler.step()
            
            print(f"Training Set - Loss: {train_loss:.4f}, Acc: {train_metrics['accuracy']:.4f}, "
                  f"F1: {train_metrics['f1']:.4f}")
            print(f"Test Set - Loss: {test_loss:.4f}, Acc: {test_metrics['accuracy']:.4f}, "
                  f"F1: {test_metrics['f1']:.4f}")
            
            if test_metrics['accuracy'] > best_test_acc:
                best_test_acc = test_metrics['accuracy']
                best_test_metrics = test_metrics
                best_train_metrics = train_metrics
                best_train_loss = train_loss
                self.model_manager.save_best_model(model)
            
            if (epoch + 1) % 10 == 0 or epoch == Config.NUM_EPOCHS - 1:
                self.model_manager.save_model(
                    model, epoch + 1, optimizer, scheduler,
                    None, best_test_metrics, best_train_metrics
                )
        
        return model, best_test_metrics, best_train_metrics
    
    def export_results(self, fold_results, test_results, train_results):
        all_results = []
        
        for fold_result in fold_results:
            fold_data = {
                'Fold': fold_result['fold'],
                'Dataset': 'Validation',
                'Loss': fold_result['val_loss'],
                'Accuracy': fold_result['val_accuracy'],
                'Precision': fold_result['val_precision'],
                'Recall': fold_result['val_recall'],
                'F1_Score': fold_result['val_f1'],
                'Train_Loss': fold_result['train_loss'],
                'Train_Accuracy': fold_result['train_accuracy'],
                'Train_F1_Score': fold_result['train_f1']
            }
            all_results.append(fold_data)
        
        if train_results:
            train_data = {
                'Fold': 'Final',
                'Dataset': 'Train',
                'Loss': train_results['loss'],
                'Accuracy': train_results['accuracy'],
                'Precision': train_results['precision'],
                'Recall': train_results['recall'],
                'F1_Score': train_results['f1'],
                'Train_Loss': train_results['loss'],
                'Train_Accuracy': train_results['accuracy'],
                'Train_F1_Score': train_results['f1']
            }
            all_results.append(train_data)
        
        if test_results:
            test_data = {
                'Fold': 'Final',
                'Dataset': 'Test',
                'Loss': test_results['loss'],
                'Accuracy': test_results['accuracy'],
                'Precision': test_results['precision'],
                'Recall': test_results['recall'],
                'F1_Score': test_results['f1'],
                'Train_Loss': train_results['loss'] if train_results else None,
                'Train_Accuracy': train_results['accuracy'] if train_results else None,
                'Train_F1_Score': train_results['f1'] if train_results else None
            }
            all_results.append(test_data)
        
        df = pd.DataFrame(all_results)
        excel_path = self.model_manager.save_dir / "training_results.xlsx"
        df.to_excel(excel_path, index=False)
        
        print(f"\n✓ Results saved to: {excel_path}")
        
        print("\n" + "="*80)
        print("Results Summary:")
        print("="*80)
        
        if fold_results:
            avg_val_acc = np.mean([r['val_accuracy'] for r in fold_results])
            print(f"5-fold cross validation average accuracy: {avg_val_acc:.4f}")
        
        if train_results:
            print(f"Final training set accuracy: {train_results['accuracy']:.4f}")
        
        if test_results:
            print(f"Test set accuracy: {test_results['accuracy']:.4f}")
            print(f"Test set F1 score: {test_results['f1']:.4f}")
        
        return df
    
    def train(self, retrain=False):
        mode = self.setup(retrain)
        
        if mode == 'resume':
            print("\nResuming training from checkpoint...")
            checkpoint = torch.load(self.model_manager.checkpoint_path, map_location=self.device)
            start_epoch = checkpoint['epoch']
            fold_results = checkpoint.get('fold_results', [])
            test_results = checkpoint.get('test_results')
            train_results = checkpoint.get('train_results')
            
            self.data_manager.split_data(Config.TEST_SIZE, Config.SEED)
            train_dataset, test_dataset = self.data_manager.create_datasets()
            
            model, test_results, train_results = self.train_final_model(
                train_dataset, test_dataset, start_epoch)
        else:
            print("\nStarting new training...")
            
            self.data_manager.split_data(Config.TEST_SIZE, Config.SEED)
            train_dataset, test_dataset = self.data_manager.create_datasets()
            
            fold_results = self.kfold_cross_validation(train_dataset)
            
            model, test_results, train_results = self.train_final_model(
                train_dataset, test_dataset)
        
        print("\n" + "="*60)
        print("Final Training Set Detailed Metrics:")
        print("="*60)
        self.utils.print_detailed_metrics(train_results, "Final Training Set")
        
        print("\n" + "="*60)
        print("Test Set Detailed Metrics:")
        print("="*60)
        self.utils.print_detailed_metrics(test_results, "Test Set")
        
        self.export_results(fold_results, test_results, train_results)
        
        self.model_manager.save_model_info(
            self.data_manager, fold_results, test_results, train_results)
        
        self.model_manager.save_model(
            model, Config.NUM_EPOCHS, None, None,
            fold_results, test_results, train_results
        )
        
        print("\n" + "="*80)
        print("Training completed! Model saved and ready for prediction.")
        print("="*80)
        
        return model, test_results, train_results, fold_results

class Predictor:
    def __init__(self, model_path=None, info_path=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.info = None
        self.transform = None
        
        if model_path is None:
            model_manager = ModelManager()
            model_path = model_manager.best_model_path
            info_path = model_manager.info_path
        
        self.load_model(model_path, info_path)
    
    def load_model(self, model_path, info_path):
        if not os.path.exists(model_path):
            print(f"Error: Model file not found {model_path}")
            return False
        
        try:
            model = ParallelResNet18()
            model.load_state_dict(torch.load(model_path, map_location=self.device))
            model = model.to(self.device)
            model.eval()
            self.model = model
            
            if os.path.exists(info_path):
                with open(info_path, 'rb') as f:
                    self.info = pickle.load(f)
                print(f"✓ Loaded model information")
            
            self.transform = transforms.Compose([
                transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])
            
            print(f"✓ Model loaded successfully: {os.path.basename(model_path)}")
            print(f"  Using device: {self.device}")
            return True
            
        except Exception as e:
            print(f"Failed to load model: {e}")
            return False
    
    def predict_single_image(self, image_path):
        if self.model is None:
            print("Error: No model loaded")
            return None, None
        
        try:
            image = Image.open(image_path).convert('RGB')
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(image_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs, 1)
                
                if self.info and 'class_names' in self.info:
                    class_names = self.info['class_names']
                else:
                    class_names = ['immature', 'mature']
                
                result = class_names[predicted.item()]
                confidence = probabilities[0][predicted.item()].item()
                
                all_probs = probabilities[0].cpu().numpy()
                
            return {
                'class': result,
                'confidence': confidence,
                'class_idx': predicted.item(),
                'probabilities': all_probs,
                'class_names': class_names
            }
            
        except Exception as e:
            print(f"Prediction failed: {e}")
            return None, None
    
    def predict_batch(self, image_paths):
        results = []
        for image_path in tqdm(image_paths, desc="Predicting"):
            result = self.predict_single_image(image_path)
            if result:
                results.append({
                    'image_path': image_path,
                    'prediction': result['class'],
                    'confidence': result['confidence'],
                    'probabilities': result['probabilities']
                })
        return results
    
    def show_prediction(self, image_path):
        result = self.predict_single_image(image_path)
        if not result:
            return
        
        print(f"\nImage: {os.path.basename(image_path)}")
        print(f"Prediction: {result['class']}")
        print(f"Confidence: {result['confidence']:.4f}")
        print(f"\nAll class probabilities:")
        for i, (class_name, prob) in enumerate(zip(result['class_names'], result['probabilities'])):
            print(f"  {class_name}: {prob:.4f}")
        
        img = Image.open(image_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(f"Prediction: {result['class']} (Confidence: {result['confidence']:.2%})")
        plt.axis('off')
        plt.show()

def main():
    print("=" * 80)
    print("Parallel ResNet18 Transfer Learning System")
    print("=" * 80)
    print("Options:")
    print("1. Train new model")
    print("2. Continue training existing model")
    print("3. Load model for prediction")
    print("4. Export training results")
    print("=" * 80)
    
    choice = input("Select (1-4): ").strip()
    
    if choice == '1':
        trainer = Trainer()
        model, test_results, train_results, fold_results = trainer.train(retrain=True)
        
        predictor = Predictor()
        print("\nTraining completed! Use predictor.predict_single_image() for predictions")
        return predictor, trainer
    
    elif choice == '2':
        trainer = Trainer()
        model, test_results, train_results, fold_results = trainer.train(retrain=False)
        
        predictor = Predictor()
        print("\nTraining completed! Use predictor.predict_single_image() for predictions")
        return predictor, trainer
    
    elif choice == '3':
        predictor = Predictor()
        
        if predictor.model is None:
            print("Failed to load model, please train first")
            return None, None
        
        test_image = input("\nEnter test image path (or press Enter to skip): ").strip()
        if test_image and os.path.exists(test_image):
            predictor.show_prediction(test_image)
        else:
            print("Model loaded, use predictor.predict_single_image() for predictions")
        
        return predictor, None
    
    elif choice == '4':
        model_manager = ModelManager()
        info = model_manager.load_model_info()
        
        if info:
            print(f"\nSaved training information found:")
            print(f"Training set size: {len(info.get('train_paths', []))}")
            print(f"Test set size: {len(info.get('test_paths', []))}")
            print(f"Training accuracy: {info.get('train_results', {}).get('accuracy', 'N/A')}")
            print(f"Test accuracy: {info.get('test_results', {}).get('accuracy', 'N/A')}")
            
            df = pd.DataFrame([{
                'Dataset': 'Training Set',
                'Accuracy': info.get('train_results', {}).get('accuracy', 0),
                'F1_Score': info.get('train_results', {}).get('f1', 0)
            }, {
                'Dataset': 'Test Set',
                'Accuracy': info.get('test_results', {}).get('accuracy', 0),
                'F1_Score': info.get('test_results', {}).get('f1', 0)
            }])
            
            export_path = model_manager.save_dir / "model_performance.xlsx"
            df.to_excel(export_path, index=False)
            print(f"\n✓ Results exported to: {export_path}")
        else:
            print("No training information found")
        
        return None, None
    
    else:
        print("Invalid selection")
        return None, None

def quick_train():
    trainer = Trainer()
    predictor, _ = trainer.train(retrain=True)
    return predictor

def quick_predict(image_path):
    predictor = Predictor()
    if predictor.model:
        return predictor.predict_single_image(image_path)
    else:
        print("Please train model first or ensure model files exist")
        return None

def load_trained_model():
    return Predictor()

if __name__ == "__main__":
    predictor, trainer = main()

Parallel ResNet18 Transfer Learning System
Options:
1. Train new model
2. Continue training existing model
3. Load model for prediction
4. Export training results


Select (1-4):  1


Parallel ResNet18 Transfer Learning Training System
✓ GPU available: NVIDIA GeForce RTX 5070 Ti
  Memory: 17.09 GB

Using device: cuda
Save directory: ./saved_models

Starting new training...
Immature images: 2980
Mature images: 1260
Total images: 4240
Train set size: 3392
Test set size: 848

Starting 5-fold cross validation...

Fold 1/5

Epoch 1/100


Training: 100%|██████████████████████████████████████████████████| 85/85 [04:38<00:00,  3.27s/it, Loss=0.473, Acc=0.72]


Train - Loss: 0.4725, Acc: 0.7870, F1: 0.7768
Val   - Loss: 0.3657, Acc: 0.8336, F1: 0.8290

Epoch 2/100


...

Training Set - Loss: 0.0995, Acc: 0.9596, F1: 0.9596
Test Set - Loss: 0.1891, Acc: 0.9328, F1: 0.9323

Final Model - Epoch 99/100


Training: 100%|██████████████████████████████████████████████████| 106/106 [05:36<00:00,  3.17s/it, Loss=0.0922, Acc=1]


Training Set - Loss: 0.0922, Acc: 0.9626, F1: 0.9624
Test Set - Loss: 0.1889, Acc: 0.9340, F1: 0.9332

Final Model - Epoch 100/100


Training: 100%|███████████████████████████████████████████████| 106/106 [05:34<00:00,  3.16s/it, Loss=0.102, Acc=0.969]


Training Set - Loss: 0.1017, Acc: 0.9596, F1: 0.9596
Test Set - Loss: 0.1828, Acc: 0.9328, F1: 0.9327
✓ Model checkpoint saved to: saved_models\parallel_resnet18_checkpoint.pth

Final Training Set Detailed Metrics:

Final Training Set Detailed Metrics:
--------------------------------------------------
Loss: 0.1084
Accuracy: 0.9540
Precision: 0.9538
Recall: 0.9540
F1-Score: 0.9539

Per-class Metrics:
  Immature (0): Precision=0.9642, Recall=0.9706, F1=0.9674
  Mature (1): Precision=0.9294, Recall=0.9147, F1=0.9220

Confusion Matrix:
[[2314   70]
 [  86  922]]

Test Set Detailed Metrics:

Test Set Detailed Metrics:
--------------------------------------------------
Loss: 0.1658
Accuracy: 0.9399
Precision: 0.9396
Recall: 0.9399
F1-Score: 0.9397

Per-class Metrics:
  Immature (0): Precision=0.9534, Recall=0.9614, F1=0.9574
  Mature (1): Precision=0.9069, Recall=0.8889, F1=0.8978

Confusion Matrix:
[[573  23]
 [ 28 224]]

✓ Results saved to: saved_models\training_results.xlsx

Results Summ